# ChronoRestore — runnable T4 training notebook
Run all cells in order with **Python 3 / T4 GPU**. This notebook embeds the complete project source, checks the pipeline, downloads the official BSDS500 starter dataset, trains a compact GAN, and evaluates it on separate test images.

This small dataset produces a starter model. Synthetic benchmark scores do not establish restoration quality on real historical photos. The final cell downloads a results ZIP to your computer.


In [ ]:
#@title 1. Write the bundled project source
from pathlib import Path
import os, subprocess, sys, json
PROJECT = Path("/content/ChronoRestore")
SOURCE_FILES = {'chronorestore/__init__.py': '"""ChronoRestore photo restoration project."""\n', 'chronorestore/data.py': '"""Deterministic paired augmentation, with separate training and validation images."""\nfrom pathlib import Path\nimport random\n\nfrom PIL import Image, ImageOps\nfrom torch.utils.data import Dataset\nfrom torchvision.transforms import functional as TF\nfrom .degradation import degrade\n\nEXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}\n\n\ndef image_paths(root):\n    paths = sorted(p.resolve() for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in EXTENSIONS)\n    if not paths:\n        raise ValueError(f"No images found in {root}. Add clean photos first.")\n    return paths\n\n\nclass PhotoDataset(Dataset):\n    def __init__(self, paths, size=256, training=True, seed=17):\n        self.paths = image_paths(paths) if isinstance(paths, (str, Path)) else list(paths)\n        if not self.paths:\n            raise ValueError("Image list is empty.")\n        self.size, self.training, self.seed, self.epoch = size, training, seed, 0\n\n    def __len__(self):\n        return len(self.paths)\n\n    def __getitem__(self, index):\n        rng = random.Random(self.seed + index + (self.epoch * len(self) if self.training else 0))\n        try:\n            with Image.open(self.paths[index]) as source:\n                image = ImageOps.exif_transpose(source).convert("RGB")\n        except (OSError, ValueError) as exc:\n            raise ValueError(f"Cannot read image {self.paths[index]}: {exc}") from exc\n        image = TF.resize(image, self.size + (32 if self.training else 0))\n        if self.training:\n            w, h = image.size\n            image = TF.crop(image, rng.randint(0, h - self.size), rng.randint(0, w - self.size), self.size, self.size)\n            if rng.random() < 0.5:\n                image = TF.hflip(image)\n        else:\n            image = TF.center_crop(image, self.size)\n        return TF.to_tensor(degrade(image, rng.randrange(2**32))), TF.to_tensor(image)\n', 'chronorestore/degradation.py': '"""Ordered synthetic photo degradation for paired training."""\n\nfrom __future__ import annotations\n\nfrom io import BytesIO\nimport random\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageEnhance\n\n\ndef degrade(image: Image.Image, seed: int | None = None) -> Image.Image:\n    """Return a reproducibly degraded RGB copy of a clean image."""\n    rng = random.Random(seed)\n    image = image.convert("RGB")\n    width, height = image.size\n\n    # Chemical aging: faded contrast and warm paper tint.\n    image = ImageEnhance.Contrast(image).enhance(rng.uniform(0.55, 0.9))\n    image = Image.blend(image, Image.new("RGB", image.size, (198, 171, 126)), rng.uniform(0.08, 0.28))\n    image = ImageEnhance.Color(image).enhance(rng.uniform(0.35, 0.8))\n\n    # Physical damage: scratches and dust blended into the print.\n    damage = Image.new("RGBA", image.size, (0, 0, 0, 0))\n    draw = ImageDraw.Draw(damage)\n    for _ in range(rng.randint(2, 12)):\n        x, y = rng.randrange(width), rng.randrange(height)\n        x2 = max(0, min(width - 1, x + rng.randint(-25, 25)))\n        y2 = max(0, min(height - 1, y + rng.randint(15, 100)))\n        draw.line((x, y, x2, y2), fill=rng.choice(((244, 226, 190, 100), (45, 35, 27, 85))), width=rng.choice((1, 1, 2)))\n    for _ in range(max(2, width * height // 18000)):\n        x, y = rng.randrange(width), rng.randrange(height)\n        radius = rng.choice((1, 1, 2, 3))\n        draw.ellipse((x - radius, y - radius, x + radius, y + radius), fill=(234, 220, 190, 110))\n    image = Image.alpha_composite(image.convert("RGBA"), damage).convert("RGB")\n\n    # Digitization: sensor noise followed by JPEG compression.\n    pixels = np.asarray(image, dtype=np.float32)\n    noise = np.random.default_rng(seed).normal(0, rng.uniform(2, 10), pixels.shape)\n    image = Image.fromarray(np.uint8(np.clip(pixels + noise, 0, 255)))\n    encoded = BytesIO()\n    image.save(encoded, format="JPEG", quality=rng.randint(35, 78))\n    encoded.seek(0)\n    image = Image.open(encoded).convert("RGB")\n\n    # Small pale tears; keep the covered area limited.\n    if rng.random() < 0.45:\n        mask = Image.new("RGBA", image.size, (0, 0, 0, 0))\n        draw = ImageDraw.Draw(mask)\n        for _ in range(rng.randint(1, 3)):\n            x, y = rng.randrange(width), rng.randrange(height)\n            rx, ry = rng.randint(3, max(4, width // 18)), rng.randint(3, max(4, height // 18))\n            draw.ellipse((x - rx, y - ry, x + rx, y + ry), fill=(220, 204, 171, 210))\n        image = Image.alpha_composite(image.convert("RGBA"), mask).convert("RGB")\n    return image\n', 'chronorestore/demo.py': '"""Upload-and-compare Gradio interface, loaded only when launched."""\nimport os\nfrom pathlib import Path\nfrom .restore import load_generator, restore_image\n\n\ndef checkpoint_path(checkpoint=None):\n    default = next((path for path in ("runs/detail_finetune/best.pt", "runs/caltech256_256px/best.pt", "runs/bsds500/best.pt", "runs/best.pt") if Path(path).is_file()), "runs/best.pt")\n    checkpoint = Path(checkpoint or os.environ.get("CHRONORESTORE_CHECKPOINT", default))\n    if not checkpoint.is_file():\n        raise FileNotFoundError(f"Checkpoint not found: {checkpoint}. Train first or set CHRONORESTORE_CHECKPOINT.")\n    return checkpoint\n\n\ndef create_app(checkpoint=None):\n    import gradio as gr\n\n    model = load_generator(checkpoint_path(checkpoint))\n\n    def run(image):\n        try:\n            return restore_image(image, model)\n        except (ValueError, RuntimeError) as exc:\n            raise gr.Error(str(exc)) from exc\n\n    with gr.Blocks(title="ChronoRestore") as app:\n        gr.Markdown("# ChronoRestore\\nRestore an old photograph and compare the result. Keep the original: reconstructed details may be approximate.")\n        with gr.Row():\n            source = gr.Image(type="pil", label="Old photo")\n            result = gr.Image(type="pil", label="Restored photo", format="png")\n        gr.Button("Restore photo", variant="primary").click(run, inputs=source, outputs=result, concurrency_limit=1)\n    return app.queue(default_concurrency_limit=1)\n\n\nif __name__ == "__main__":\n    import uvicorn\n    from .studio import create_studio\n\n    uvicorn.run(create_studio(), host="127.0.0.1", port=7860)\n', 'chronorestore/download.py': '"""Download the official BSDS500 starter dataset, retaining its image splits."""\nimport argparse\nimport hashlib\nfrom pathlib import Path\nimport shutil\nimport tarfile\nimport urllib.request\n\nURL = "https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/BSR/BSR_bsds500.tgz"\n\n\ndef download(destination):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    archive = destination / "BSR_bsds500.tgz"\n    if not archive.is_file():\n        temporary = archive.with_suffix(".part")\n        print(f"Downloading {URL}", flush=True)\n        with urllib.request.urlopen(URL, timeout=60) as response, temporary.open("wb") as target:\n            shutil.copyfileobj(response, target)\n        temporary.replace(archive)\n    counts = {split: 0 for split in ("train", "val", "test")}\n    with tarfile.open(archive) as bundle:\n        for member in bundle:\n            parts = Path(member.name).parts\n            if member.isfile() and len(parts) >= 3 and parts[-3] == "images" and parts[-2] in counts and Path(parts[-1]).suffix.lower() == ".jpg":\n                split, name = parts[-2:]\n                folder = destination / split\n                folder.mkdir(exist_ok=True)\n                with bundle.extractfile(member) as source, (folder / name).open("wb") as target:\n                    shutil.copyfileobj(source, target)\n                counts[split] += 1\n    if counts != {"train": 200, "val": 100, "test": 200}:\n        raise RuntimeError(f"Unexpected BSDS500 image counts: {counts}. Check the downloaded archive.")\n    digest = hashlib.sha256(archive.read_bytes()).hexdigest()\n    (destination / "SOURCE.txt").write_text(f"Source: {URL}\\nSHA256: {digest}\\nCitation: Arbelaez et al., Contour Detection and Hierarchical Image Segmentation, TPAMI 2011.\\n")\n    print(f"Ready: {destination} — {counts}", flush=True)\n    return destination\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output", default="data/bsds500")\n    download(parser.parse_args().output)\n', 'chronorestore/download_large.py': '"""Prepare a reproducible 5,000-image training set from official Caltech256."""\nimport hashlib\nfrom io import BytesIO\nimport json\nfrom pathlib import Path\nimport random\nimport tarfile\nimport urllib.request\n\nfrom PIL import Image, ImageOps\n\nURL = "https://data.caltech.edu/records/nyy15-4j048/files/256_ObjectCategories.tar"\nMD5 = "67b4f42ca05d46448c6bb8ecd2220f6d"\n\n\ndef prepare(destination="data/caltech256_restoration"):\n    root = Path(destination)\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / "manifest.json"\n    if marker.is_file():\n        manifest = json.loads(marker.read_text())\n        if all((root / row["path"]).is_file() for row in manifest["images"]):\n            print(f"Dataset already prepared: {root}", flush=True)\n            return root\n    archive = root / "256_ObjectCategories.tar"\n    if not archive.is_file():\n        temporary = archive.with_suffix(".part")\n        print("Downloading Caltech256 (about 1.2 GB) from the official repository...", flush=True)\n        with urllib.request.urlopen(urllib.request.Request(URL + "?download=1", headers={"User-Agent": "curl/8.7.1"}), timeout=120) as response, temporary.open("wb") as target:\n            downloaded = 0\n            while chunk := response.read(4 * 1024 * 1024):\n                target.write(chunk)\n                downloaded += len(chunk)\n                if downloaded % (64 * 1024 * 1024) == 0:\n                    print(f"Downloaded {downloaded // (1024 * 1024)} MiB", flush=True)\n        temporary.replace(archive)\n    digest = hashlib.md5()\n    with archive.open("rb") as source:\n        while chunk := source.read(4 * 1024 * 1024):\n            digest.update(chunk)\n    if digest.hexdigest() != MD5:\n        raise RuntimeError("Archive checksum mismatch. Remove the incomplete archive and retry.")\n    rows, seen = [], set()\n    for split in ("train", "val", "test"):\n        (root / split).mkdir(exist_ok=True)\n    with tarfile.open(archive) as bundle:\n        members = [member for member in bundle.getmembers() if member.isfile() and member.name.lower().endswith(\'.jpg\')]\n        random.Random(17).shuffle(members)\n        for member in members:\n            try:\n                with bundle.extractfile(member) as stream, Image.open(BytesIO(stream.read())) as source:\n                    image = ImageOps.exif_transpose(source).convert(\'RGB\')\n                    w, h = image.size\n                    if min(w, h) < 256 or max(w, h) / min(w, h) > 3:\n                        continue\n                    scale = 320 / min(w, h)\n                    image = image.resize((round(w * scale), round(h * scale)), Image.Resampling.LANCZOS)\n            except (OSError, ValueError):\n                continue\n            fingerprint = hashlib.sha256(image.tobytes()).hexdigest()\n            if fingerprint in seen:\n                continue\n            seen.add(fingerprint)\n            split = "train" if len(rows) < 5000 else "val" if len(rows) < 5500 else "test"\n            relative = Path(split) / f"{len(rows):05d}.jpg"\n            image.save(root / relative, quality=95)\n            rows.append({"path": str(relative), "source": member.name, "pixel_sha256": fingerprint})\n            if len(rows) % 500 == 0:\n                print(f"Prepared {len(rows)}/6000 photos", flush=True)\n            if len(rows) == 6000:\n                break\n    if len(rows) != 6000:\n        raise RuntimeError(f"Only {len(rows)} eligible photos found; refusing an incomplete split.")\n    marker.write_text(json.dumps({"source": URL, "archive_md5": MD5, "seed": 17, "counts": {"train": 5000, "val": 500, "test": 500}, "images": rows}, indent=2))\n    print(f"Ready: {root} — 5000 train, 500 validation, 500 test", flush=True)\n    return root\n\n\nif __name__ == "__main__":\n    prepare()\n', 'chronorestore/evaluate.py': '"""Report paired synthetic PSNR/SSIM on a separate image folder."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom skimage.metrics import peak_signal_noise_ratio, structural_similarity\nfrom PIL import Image, ImageOps\nfrom torchvision.transforms import functional as TF\n\nfrom .data import image_paths\nfrom .degradation import degrade\nfrom .restore import load_generator, restore_image\nfrom .runtime import get_device\n\n\ndef scores(prediction, target):\n    prediction, target = np.asarray(prediction, dtype=np.float32) / 255, np.asarray(target, dtype=np.float32) / 255\n    return (float(peak_signal_noise_ratio(target, prediction, data_range=1)),\n            float(structural_similarity(target, prediction, data_range=1, channel_axis=-1)))\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--checkpoint", required=True)\n    parser.add_argument("--data", required=True, help="Held-out images; never use the training folder")\n    parser.add_argument("--count", type=int, default=200)\n    parser.add_argument("--size", type=int, default=128)\n    parser.add_argument("--seed", type=int, default=1701)\n    parser.add_argument("--output", default="evaluation")\n    parser.add_argument("--device", default="auto", choices=("auto", "cpu", "cuda", "mps"))\n    args = parser.parse_args(argv)\n    if args.count < 1 or args.size < 32:\n        parser.error("count must be positive and size must be >=32")\n    paths = image_paths(args.data)[:args.count]\n    manifest = Path(args.checkpoint).parent / "split.json"\n    if manifest.is_file():\n        used = json.loads(manifest.read_text())["train"]\n        if set(map(str, paths)) & set(used):\n            parser.error("Evaluation contains training images. Choose a held-out folder.")\n    model = load_generator(args.checkpoint, get_device(args.device))\n    output = Path(args.output)\n    output.mkdir(parents=True, exist_ok=True)\n    rows = []\n    for i, path in enumerate(paths):\n        with Image.open(path) as image:\n            clean = TF.center_crop(TF.resize(ImageOps.exif_transpose(image).convert("RGB"), args.size), args.size)\n        damaged = degrade(clean, args.seed + i)\n        restored = restore_image(damaged, model)\n        before, after = scores(damaged, clean), scores(restored, clean)\n        rows.append({"image": str(path), "input_psnr": before[0], "input_ssim": before[1], "restored_psnr": after[0], "restored_ssim": after[1]})\n        if i < 6:\n            gallery = Image.new("RGB", (args.size * 3, args.size))\n            for column, photo in enumerate((damaged, restored, clean)):\n                gallery.paste(photo, (column * args.size, 0))\n            gallery.save(output / f"comparison_{i+1}.png")\n    report = {"count": len(rows), "seed": args.seed, "size": args.size,\n              "mean": {key: float(np.mean([row[key] for row in rows])) for key in rows[0] if key != "image"}, "images": rows}\n    (output / "metrics.json").write_text(json.dumps(report, indent=2))\n    print(json.dumps(report["mean"], indent=2))\n    print(f"Saved metrics and galleries (damaged / restored / clean) to {output}. Synthetic benchmark only.")\n\n\nif __name__ == "__main__":\n    main()\n', 'chronorestore/model.py': '"""Compact U-Net generator and PatchGAN discriminator."""\n\nfrom __future__ import annotations\n\nimport torch\nfrom torch import nn\nfrom torch.nn.utils import spectral_norm\n\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int, down: bool = False):\n        super().__init__()\n        if down:\n            layers: list[nn.Module] = [nn.Conv2d(in_channels, out_channels, 4, 2, 1)]\n        else:\n            layers = [nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(in_channels, out_channels, 3, 1, 1)]\n        layers += [nn.InstanceNorm2d(out_channels), nn.LeakyReLU(0.2, inplace=True)]\n        self.layers = nn.Sequential(*layers)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.layers(x)\n\n\nclass Generator(nn.Module):\n    def __init__(self, base: int = 32):\n        super().__init__()\n        self.in_conv = nn.Sequential(nn.Conv2d(3, base, 3, 1, 1), nn.LeakyReLU(0.2, inplace=True))\n        self.down1 = ConvBlock(base, base * 2, down=True)\n        self.down2 = ConvBlock(base * 2, base * 4, down=True)\n        self.down3 = ConvBlock(base * 4, base * 8, down=True)\n        self.down4 = ConvBlock(base * 8, base * 8, down=True)\n        self.up1 = ConvBlock(base * 8, base * 8)\n        self.up2 = ConvBlock(base * 16, base * 4)\n        self.up3 = ConvBlock(base * 8, base * 2)\n        self.up4 = ConvBlock(base * 4, base)\n        self.out = nn.Sequential(nn.Conv2d(base * 2, 3, 3, 1, 1), nn.Sigmoid())\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        a = self.in_conv(x)\n        b = self.down1(a)\n        c = self.down2(b)\n        d = self.down3(c)\n        e = self.down4(d)\n        x = self.up1(e)\n        x = self.up2(torch.cat((x, d), dim=1))\n        x = self.up3(torch.cat((x, c), dim=1))\n        x = self.up4(torch.cat((x, b), dim=1))\n        return self.out(torch.cat((x, a), dim=1))\n\n\nclass Discriminator(nn.Module):\n    def __init__(self, base: int = 32):\n        super().__init__()\n        self.net = nn.Sequential(\n            spectral_norm(nn.Conv2d(6, base, 4, 2, 1)), nn.LeakyReLU(0.2, inplace=True),\n            spectral_norm(nn.Conv2d(base, base * 2, 4, 2, 1)), nn.InstanceNorm2d(base * 2), nn.LeakyReLU(0.2, inplace=True),\n            spectral_norm(nn.Conv2d(base * 2, base * 4, 4, 2, 1)), nn.InstanceNorm2d(base * 4), nn.LeakyReLU(0.2, inplace=True),\n            spectral_norm(nn.Conv2d(base * 4, 1, 3, 1, 1)),\n        )\n\n    def forward(self, degraded: torch.Tensor, restored: torch.Tensor) -> torch.Tensor:\n        return self.net(torch.cat((degraded, restored), dim=1))\n', 'chronorestore/restore.py': '"""Restore photos at their original size with bounded GPU memory."""\nimport argparse\nfrom functools import lru_cache\nfrom itertools import islice, product\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageOps\nfrom torchvision.transforms import functional as TF\n\nfrom .model import Generator\nfrom .runtime import get_device\n\n\ndef load_generator(checkpoint, device=None):\n    device = get_device() if device is None else torch.device(device)\n    state = torch.load(checkpoint, map_location="cpu", weights_only=True)\n    model = Generator().to(device)\n    model.load_state_dict(state.get("generator", state))\n    return model.eval()\n\n\ndef _positions(length, tile, overlap):\n    return list(range(0, max(1, length - tile + 1), tile - overlap)) + ([length - tile] if length > tile and (length - tile) % (tile - overlap) else [])\n\n\n@lru_cache(maxsize=16)\ndef _window(height, width):\n    return np.maximum(np.outer(np.hanning(height), np.hanning(width)), 0.01).astype(np.float32)\n\n\n@torch.inference_mode()\ndef restore_image(image, model, device=None, tile_size=256, overlap=32, refine=False):\n    if not isinstance(image, Image.Image):\n        raise ValueError("Provide a readable image.")\n    if tile_size < 32 or tile_size % 16 or overlap < 0 or overlap >= tile_size:\n        raise ValueError("tile_size must be >=32 and divisible by 16; overlap must be smaller than tile_size.")\n    device = next(model.parameters()).device if device is None else torch.device(device)\n    if image.width * image.height > 25_000_000:\n        raise ValueError("Resize photos to at most 25 megapixels before restoring.")\n    image = ImageOps.exif_transpose(image).convert("RGB")\n    width, height = image.size\n    pixels = TF.to_tensor(image)\n    output = np.zeros((3, height, width), dtype=np.float32)\n    weights = np.zeros((height, width), dtype=np.float32)\n    model.eval()\n\n    h, w = min(height, tile_size), min(width, tile_size)\n    pad = (0, max(32, (w + 15) // 16 * 16) - w, 0, max(32, (h + 15) // 16 * 16) - h)\n    window = _window(h, w)\n    positions = product(_positions(height, tile_size, overlap), _positions(width, tile_size, overlap))\n    batch_size = 4 if device.type in ("cuda", "mps") and tile_size <= 256 else 1\n    while batch := list(islice(positions, batch_size)):\n        patches = torch.stack([pixels[:, y:y+h, x:x+w] for y, x in batch]).to(device)\n        if any(pad):\n            patches = torch.nn.functional.pad(patches, pad, mode="replicate")\n        prediction = model(patches)\n        if refine:\n            for axes in ((-1,), (-2,), (-2, -1)):\n                prediction += model(patches.flip(axes)).flip(axes)\n            prediction /= 4\n        restored = prediction[:, :, :h, :w].cpu().numpy()\n        for (y, x), tile in zip(batch, restored):\n            output[:, y:y+h, x:x+w] += tile * window\n            weights[y:y+h, x:x+w] += window\n    output /= weights[None]\n    return Image.fromarray(np.uint8(np.clip(output.transpose(1, 2, 0) * 255, 0, 255).round()))\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--checkpoint", required=True)\n    parser.add_argument("--input", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--device", default="auto", choices=("auto", "cpu", "cuda", "mps"))\n    args = parser.parse_args(argv)\n    source, output = Path(args.input), Path(args.output)\n    if not source.is_file() or not Path(args.checkpoint).is_file():\n        parser.error("Input image and checkpoint must exist.")\n    if source.resolve() == output.resolve() or output.exists():\n        parser.error("Choose a new output filename to preserve existing photos.")\n    model = load_generator(args.checkpoint, get_device(args.device))\n    with Image.open(source) as image:\n        restored = restore_image(image, model)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    restored.save(output)\n    print(f"Saved {output}")\n\n\nif __name__ == "__main__":\n    main()\n', 'chronorestore/runtime.py': '"""Shared device selection."""\nimport torch\n\n\ndef get_device(name="auto"):\n    if name != "auto":\n        return torch.device(name)\n    if torch.cuda.is_available():\n        return torch.device("cuda")\n    if torch.backends.mps.is_available():\n        return torch.device("mps")\n    return torch.device("cpu")\n', 'chronorestore/studio.py': '"""Local photo restoration studio, backed by the existing PyTorch model."""\nimport asyncio\nfrom io import BytesIO\nfrom pathlib import Path\n\nfrom fastapi import FastAPI, HTTPException, Request\nfrom fastapi.responses import FileResponse, Response\nfrom fastapi.staticfiles import StaticFiles\nfrom PIL import Image, UnidentifiedImageError\nfrom starlette.concurrency import run_in_threadpool\n\nfrom .demo import checkpoint_path\nfrom .restore import load_generator, restore_image\n\nASSETS = Path(__file__).with_name("studio_assets")\nMAX_BYTES = 24 * 1024 * 1024\nMAX_PIXELS = 25_000_000\n\n\ndef create_studio(checkpoint=None):\n    model = load_generator(checkpoint_path(checkpoint))\n    lock = asyncio.Lock()\n    app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)\n    app.mount("/studio", StaticFiles(directory=ASSETS), name="studio")\n\n    @app.get("/")\n    def home():\n        return FileResponse(ASSETS / "archive.html", headers={"Cache-Control": "no-cache"})\n\n    @app.get("/workbench")\n    def workbench():\n        return FileResponse(ASSETS / "index.html", headers={"Cache-Control": "no-cache"})\n\n    def render(upload, refine):\n        try:\n            with Image.open(upload, formats=["JPEG", "PNG", "WEBP"]) as photo:\n                if photo.width * photo.height > MAX_PIXELS:\n                    raise HTTPException(413, "Choose an image of 25 megapixels or less.")\n                result = restore_image(photo, model, refine=refine)\n        except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombError) as exc:\n            raise HTTPException(400, "Choose a readable JPEG, PNG, or WebP image.") from exc\n        output = BytesIO()\n        result.save(output, format="PNG", compress_level=1)\n        return output.getvalue()\n\n    @app.post("/api/restore")\n    async def restore(request: Request, refine: bool = False):\n        if request.headers.get("content-type") != "application/octet-stream":\n            raise HTTPException(415, "Send the image as application/octet-stream.")\n        if lock.locked():\n            raise HTTPException(503, "The studio is busy. Try again in a moment.")\n        async with lock:\n            upload = BytesIO()\n            async for chunk in request.stream():\n                if upload.tell() + len(chunk) > MAX_BYTES:\n                    raise HTTPException(413, "Choose a file no larger than 24 MB.")\n                upload.write(chunk)\n            upload.seek(0)\n            result = await run_in_threadpool(render, upload, refine)\n        return Response(result, media_type="image/png", headers={"Cache-Control": "no-store"})\n\n    return app\n', 'chronorestore/train.py': '"""Train, validate, and resume the compact restoration GAN."""\nimport argparse\nimport csv\nimport json\nimport math\nfrom pathlib import Path\nimport random\n\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader\n\nfrom .data import PhotoDataset, image_paths\nfrom .model import Discriminator, Generator\nfrom .runtime import get_device\n\n\ndef atomic_save(state, path):\n    temporary = path.with_suffix(".tmp")\n    torch.save(state, temporary)\n    temporary.replace(path)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--data", default="data/clean")\n    parser.add_argument("--val-data", help="Separate validation folder; otherwise reserve 10%% of data")\n    parser.add_argument("--output", default="runs")\n    parser.add_argument("--epochs", type=int, default=30, help="Total epochs, including resumed epochs")\n    parser.add_argument("--size", type=int, default=128)\n    parser.add_argument("--batch-size", type=int, default=8)\n    parser.add_argument("--seed", type=int, default=17)\n    parser.add_argument("--device", default="auto", choices=("auto", "cpu", "cuda", "mps"))\n    parser.add_argument("--resume", help="Path to latest.pt from an interrupted run")\n    parser.add_argument("--warmup-epochs", type=int, default=2)\n    parser.add_argument("--gan-weight", type=float, default=0.01)\n    parser.add_argument("--patience", type=int, default=0, help="Stop after this many epochs without validation improvement; 0 disables")\n    args = parser.parse_args(argv)\n    if args.epochs < 1 or args.batch_size < 1 or args.size < 32 or args.size % 16:\n        parser.error("epochs and batch-size must be positive; size must be >=32 and divisible by 16")\n    if args.patience < 0 or args.warmup_epochs < 0 or not math.isfinite(args.gan_weight) or args.gan_weight < 0:\n        parser.error("patience, warmup-epochs and gan-weight must be nonnegative")\n\n    random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    device = get_device(args.device)\n    output = Path(args.output)\n    output.mkdir(parents=True, exist_ok=True)\n    if not args.resume and any((output / name).exists() for name in ("latest.pt", "best.pt", "losses.csv")):\n        parser.error("Output already contains a run. Use --resume or a different --output.")\n    paths = image_paths(args.data)\n    if args.val_data:\n        train_paths, val_paths = paths, image_paths(args.val_data)\n        if set(train_paths) & set(val_paths):\n            parser.error("Training and validation folders overlap.")\n    else:\n        if len(paths) < 2:\n            parser.error("Need at least two images to separate training and validation.")\n        random.Random(args.seed).shuffle(paths)\n        n_val = max(1, len(paths) // 10)\n        val_paths, train_paths = paths[:n_val], paths[n_val:]\n    split = {"train": [str(p) for p in train_paths], "validation": [str(p) for p in val_paths]}\n    train_data = PhotoDataset(train_paths, args.size, True, args.seed)\n    val_data = PhotoDataset(val_paths, args.size, False, args.seed + 100000)\n    shuffle_rng = torch.Generator()\n    loader = DataLoader(train_data, batch_size=args.batch_size, shuffle=True, generator=shuffle_rng, num_workers=0)\n    val_loader = DataLoader(val_data, batch_size=args.batch_size, num_workers=0)\n    generator, discriminator = Generator().to(device), Discriminator().to(device)\n    opt_g = torch.optim.Adam(generator.parameters(), lr=1e-4, betas=(0.5, 0.999))\n    opt_d = torch.optim.Adam(discriminator.parameters(), lr=4e-4, betas=(0.5, 0.999))\n    l1, bce = nn.L1Loss(), nn.BCEWithLogitsLoss()\n    start, best, history = 1, float("inf"), []\n    config = {key: getattr(args, key) for key in ("size", "seed", "batch_size", "warmup_epochs", "gan_weight")}\n    if args.resume:\n        state = torch.load(args.resume, map_location=device, weights_only=True)\n        if state.get("config") != config or state.get("split") != split:\n            parser.error("Resume requires the same dataset paths, size, seed, batch size, warmup, and GAN weight.")\n        generator.load_state_dict(state["generator"])\n        discriminator.load_state_dict(state["discriminator"])\n        opt_g.load_state_dict(state["optimizer_g"])\n        opt_d.load_state_dict(state["optimizer_d"])\n        start, best, history = state["epoch"] + 1, state["best_val_l1"], state["history"]\n    (output / "split.json").write_text(json.dumps(split, indent=2))\n    print(f"Device: {device} | train: {len(train_data)} | validation: {len(val_data)}", flush=True)\n    for epoch in range(start, args.epochs + 1):\n        train_data.epoch = epoch\n        shuffle_rng.manual_seed(args.seed + epoch)\n        generator.train()\n        totals = [0.0, 0.0, 0.0]\n        adversarial = epoch > args.warmup_epochs and args.gan_weight > 0\n        for step, (damaged, clean) in enumerate(loader, 1):\n            damaged, clean = damaged.to(device), clean.to(device)\n            restored = generator(damaged)\n            d_loss = torch.zeros((), device=device)\n            if adversarial:\n                discriminator.train().requires_grad_(True)\n                opt_d.zero_grad(set_to_none=True)\n                real_logits = discriminator(damaged, clean)\n                fake_logits = discriminator(damaged, restored.detach())\n                d_loss = (bce(real_logits, torch.full_like(real_logits, 0.95)) + bce(fake_logits, torch.zeros_like(fake_logits))) / 2\n                if not torch.isfinite(d_loss):\n                    raise RuntimeError("Non-finite discriminator loss; previous checkpoint is preserved.")\n                d_loss.backward()\n                opt_d.step()\n            opt_g.zero_grad(set_to_none=True)\n            pixel_loss = l1(restored, clean)\n            g_loss = pixel_loss\n            if adversarial:\n                discriminator.eval().requires_grad_(False)\n                logits = discriminator(damaged, restored)\n                g_loss = g_loss + args.gan_weight * bce(logits, torch.ones_like(logits))\n            if not torch.isfinite(g_loss):\n                raise RuntimeError("Non-finite generator loss; previous checkpoint is preserved.")\n            g_loss.backward()\n            opt_g.step()\n            for i, loss in enumerate((g_loss, d_loss, pixel_loss)):\n                totals[i] += loss.item() * len(clean)\n            if step == 1 or step % 100 == 0:\n                print(f"Epoch {epoch:03d} | batch {step}/{len(loader)} | L1 {pixel_loss.item():.4f}", flush=True)\n        generator.eval()\n        val_loss = 0.0\n        with torch.inference_mode():\n            for damaged, clean in val_loader:\n                val_loss += l1(generator(damaged.to(device)), clean.to(device)).item() * len(clean)\n        val_loss /= len(val_data)\n        if not math.isfinite(val_loss):\n            raise RuntimeError("Non-finite validation loss; previous checkpoint is preserved.")\n        row = {"epoch": epoch, "g_loss": totals[0] / len(train_data), "d_loss": totals[1] / len(train_data), "train_l1": totals[2] / len(train_data), "val_l1": val_loss}\n        history.append(row)\n        improved = val_loss < best\n        best = min(best, val_loss)\n        state = {"generator": generator.state_dict(), "discriminator": discriminator.state_dict(), "optimizer_g": opt_g.state_dict(), "optimizer_d": opt_d.state_dict(), "epoch": epoch, "config": config, "split": split, "history": history, "best_val_l1": best}\n        atomic_save(state, output / "latest.pt")\n        if improved:\n            atomic_save({"generator": generator.state_dict(), "epoch": epoch, "config": config, "val_l1": val_loss}, output / "best.pt")\n        with (output / "losses.csv").open("w", newline="") as f:\n            writer = csv.DictWriter(f, fieldnames=list(row))\n            writer.writeheader()\n            writer.writerows(history)\n        print(f"Epoch {epoch:03d}/{args.epochs} | train L1 {row[\'train_l1\']:.4f} | val L1 {val_loss:.4f} | GAN {\'on\' if adversarial else \'warmup\'}", flush=True)\n        best_epoch = min(history, key=lambda record: record["val_l1"])["epoch"]\n        if args.patience and epoch - best_epoch >= args.patience:\n            print(f"Early stopping: best validation was epoch {best_epoch}.", flush=True)\n            break\n    print(f"Checkpoints: {output}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'tests/test_pipeline.py': '"""Regression checks for the complete training-to-restoration path."""\nimport json\nfrom pathlib import Path\nimport tempfile\nimport unittest\n\nimport numpy as np\nfrom PIL import Image\nimport torch\n\nfrom chronorestore.data import PhotoDataset\nfrom chronorestore.degradation import degrade\nfrom chronorestore.model import Generator, Discriminator\nfrom chronorestore.restore import load_generator, restore_image, main as restore_main\nfrom chronorestore.train import main as train_main\nfrom chronorestore.evaluate import main as evaluate_main\n\n\ndef photo(seed, size=(48, 40)):\n    pixels = np.random.default_rng(seed).integers(0, 256, (*size[::-1], 3), dtype=np.uint8)\n    return Image.fromarray(pixels)\n\n\nclass Identity(torch.nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.weight = torch.nn.Parameter(torch.tensor(1.0))\n\n    def forward(self, x):\n        return x * self.weight\n\n\nclass PipelineTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(1)\n\n    def test_degradation_is_repeatable_and_preserves_input(self):\n        image = photo(1)\n        original = np.array(image)\n        a, b = degrade(image, 17), degrade(image, 17)\n        np.testing.assert_array_equal(a, b)\n        np.testing.assert_array_equal(image, original)\n        self.assertEqual(a.size, image.size)\n        self.assertFalse(np.array_equal(a, original))\n\n    def test_models_support_training_patch_sizes(self):\n        generator, discriminator = Generator(), Discriminator()\n        for size in (32, 64, 128):\n            x = torch.rand(1, 3, size, size)\n            y = generator(x)\n            self.assertEqual(y.shape, x.shape)\n            self.assertTrue(torch.isfinite(discriminator(x, y)).all())\n\n    def test_tiling_preserves_every_pixel_including_tiny_images(self):\n        for size in ((1, 1), (7, 3), (31, 17), (64, 65), (189, 97)):\n            image = photo(2, size)\n            restored = restore_image(image, Identity(), tile_size=64, overlap=16)\n            np.testing.assert_array_equal(image, restored)\n\n    def test_inference_rejects_invalid_settings(self):\n        with self.assertRaises(ValueError):\n            restore_image(None, Identity())\n        for tile, overlap in ((15, 0), (64, 64), (64, -1)):\n            with self.assertRaises(ValueError):\n                restore_image(photo(1), Identity(), tile_size=tile, overlap=overlap)\n\n    def test_dataset_reports_corrupt_file(self):\n        with tempfile.TemporaryDirectory() as root:\n            path = Path(root) / "broken.jpg"\n            path.write_text("not an image")\n            with self.assertRaisesRegex(ValueError, "broken.jpg"):\n                PhotoDataset(root)[0]\n\n    def test_early_stopping_saves_the_last_and_best_epochs(self):\n        from unittest.mock import patch\n\n        class ConstantLoss(torch.nn.Module):\n            def forward(self, prediction, target):\n                return prediction.sum() * 0 + 1\n\n        with tempfile.TemporaryDirectory() as directory:\n            root = Path(directory)\n            data = root / "photos"\n            data.mkdir()\n            for i in range(3):\n                photo(i).save(data / f"{i}.png")\n            with patch("chronorestore.train.nn.L1Loss", return_value=ConstantLoss()):\n                train_main(["--data", str(data), "--output", str(root / "run"), "--epochs", "5", "--patience", "1", "--gan-weight", "0", "--size", "32", "--device", "cpu"])\n            state = torch.load(root / "run/latest.pt", weights_only=True)\n            best = torch.load(root / "run/best.pt", weights_only=True)\n            self.assertEqual(state["epoch"], 2)\n            self.assertEqual(best["epoch"], 1)\n\n    def test_train_resume_restore_evaluate_and_demo(self):\n        with tempfile.TemporaryDirectory() as root:\n            root = Path(root)\n            train, val, test = (root / split for split in ("train", "val", "test"))\n            for folder in (train, val, test):\n                folder.mkdir()\n                for i in range(2):\n                    photo(i + len(str(folder))).save(folder / f"{i}.png")\n            run, uninterrupted = root / "run", root / "uninterrupted"\n            common = ["--data", str(train), "--val-data", str(val), "--size", "32", "--batch-size", "2", "--device", "cpu", "--warmup-epochs", "0"]\n            train_main(common + ["--output", str(run), "--epochs", "1"])\n            train_main(common + ["--output", str(run), "--epochs", "2", "--resume", str(run / "latest.pt")])\n            train_main(common + ["--output", str(uninterrupted), "--epochs", "2"])\n            resumed = torch.load(run / "latest.pt", weights_only=True)\n            complete = torch.load(uninterrupted / "latest.pt", weights_only=True)\n            self.assertEqual(resumed["epoch"], 2)\n            self.assertEqual(len(resumed["history"]), 2)\n            for name, weights in resumed["generator"].items():\n                torch.testing.assert_close(weights, complete["generator"][name], atol=0, rtol=0)\n            model = load_generator(run / "best.pt", "cpu")\n            self.assertEqual(restore_image(photo(3, (17, 19)), model).size, (17, 19))\n            result = root / "restored.png"\n            restore_main(["--checkpoint", str(run / "best.pt"), "--input", str(test / "0.png"), "--output", str(result), "--device", "cpu"])\n            self.assertTrue(result.is_file())\n            with self.assertRaises(SystemExit):\n                restore_main(["--checkpoint", str(run / "best.pt"), "--input", str(test / "0.png"), "--output", str(result)])\n            evaluation = root / "evaluation"\n            evaluate_main(["--checkpoint", str(run / "best.pt"), "--data", str(test), "--output", str(evaluation), "--size", "32", "--device", "cpu"])\n            report = json.loads((evaluation / "metrics.json").read_text())\n            self.assertEqual(report["count"], 2)\n            self.assertTrue(all(np.isfinite(v) for v in report["mean"].values()))\n            with self.assertRaises(SystemExit):\n                evaluate_main(["--checkpoint", str(run / "best.pt"), "--data", str(train)])\n            with self.assertRaises(SystemExit):\n                train_main(common + ["--output", str(run)])\n            from chronorestore.demo import create_app\n            app = create_app(run / "best.pt")\n            self.assertTrue(app.config["components"])\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'README.md': '# ChronoRestore\n\nA compact old-photo restoration GAN: synthetic aging and damage → U-Net restoration → comparison with the original. The project includes training, resumable checkpoints, held-out evaluation, image restoration, and a local restoration studio.\n\n## Ready-to-use larger model\n\nThe ready bundle includes the completed 5,000-image, 256px training run. All 30 epochs finished; epoch 28 had the best validation loss. On 500 held-out synthetic test images, PSNR improved from 18.54 to 26.37 dB and SSIM from 0.7305 to 0.8199. Real historical-photo quality still needs visual evaluation.\n\nLaunch from the project folder:\n\n```bash\npython -m chronorestore.demo\n```\n\nOpen the [ChronoRestore Archive](http://127.0.0.1:7860/) to rotate the photo cube and browse damaged, restored, and clean references. Choose a collection case or open the [restoration studio](http://127.0.0.1:7860/workbench) to upload a JPG, PNG, or WebP image. Choose a file or a BSDS500 dataset sample and the studio automatically opens the live before-and-after slider. Switch between Before and After, or save a PNG copy at the original image size. The studio runs locally and supports up to 24 MB and 25 megapixels per upload.\n\nThe studio selects `runs/detail_finetune/best.pt` when present, then falls back to the Caltech256 checkpoint. See [Detail benchmark](docs/QUALITY_BENCHMARK.md) for the paired results displayed in the studio. An explicit checkpoint or `CHRONORESTORE_CHECKPOINT` overrides this choice. The original starter model is preserved. See `docs/LARGER_TRAINING.md`; use `notebooks/ChronoRestore_256px.ipynb` to reproduce the larger run.\n\n## Ready-to-use trained starter\n\nThe ready bundle includes `runs/bsds500/best.pt` (selected at epoch 28 from a 30-epoch Tesla T4 run), the resumable `latest.pt`, logs, and `evaluation/metrics.json`. On 200 held-out synthetic examples, mean PSNR improved from 18.66 to 21.43 dB and mean SSIM from 0.689 to 0.723. Some damage remains, and performance on real historical scans is unverified. See `docs/VERIFICATION.md` for exact checks and limits.\n\nAfter installing the requirements, launch the trained model with:\n\n```bash\nCHRONORESTORE_CHECKPOINT=runs/bsds500/best.pt python -m chronorestore.demo\n```\n\nOn Windows PowerShell, set `$env:CHRONORESTORE_CHECKPOINT="runs/bsds500/best.pt"` before running `python -m chronorestore.demo`.\n\nThe archive cube keeps rotating and changes direction over time. Choose Damaged, Restored, or Clean Reference beneath the cube. Drag it or use the arrow keys to change the angle. Press Space while the cube has focus to pause or resume. The desktop site uses a camera pointer with a colored ink trail and click splash. The proof sheet shows the recorded project scores for three individual synthetic test images. Lenis 1.3.27 is served locally on the archive page, with its MIT license in `chronorestore/studio_assets/lenis.LICENSE`. Reduced motion uses native scrolling and disables automatic rotation.\n\n## Original checkpoint benchmark\n\nThis earlier experiment compares the same 200 BSDS500 test crops with identical synthetic damage. Median filtering plus autocontrast reached 20.95 dB PSNR and 0.7810 SSIM. Standard reached 25.44 dB and 0.7934. Refine averaged four predictions in flipped orientations and reached 26.03 dB and 0.8232. It improved both metrics on all 200 images relative to Standard. These are synthetic results and provide no comparison to published models on a shared benchmark.\n\nReproduce with `python -m scripts.benchmark_studio` after downloading BSDS500 and placing the trained Caltech256 checkpoint in `runs/caltech256_256px/best.pt`. The benchmark runs on 256-pixel crops and writes its data to `chronorestore/studio_assets/benchmark.json`. On the measured Apple MPS device, median processing times were 14.7 ms for Standard and 52.5 ms for Refine. Upload, PNG encoding and browser rendering are outside that timing.\n\n## Google Colab training\n\nOpen `notebooks/ChronoRestore_Colab.ipynb` in Colab, select **Python 3 / T4 GPU**, and run all cells. The notebook contains the source code, installs its extra dependencies, runs regression checks, downloads the official BSDS500 starter dataset, trains for 30 epochs, evaluates on its test split, and downloads a results ZIP. No source ZIP or personal photos are needed to start.\n\nDefaults: 128-pixel crops, batch size 8, two reconstruction-only warmup epochs, then L1 + adversarial training. These are conservative settings for a free T4. Free GPU availability and session lifetimes are controlled by Colab.\n\n`USE_DRIVE=False` avoids any Drive authorization. Results remain in the current runtime until the final ZIP downloads; an expired/deleted runtime loses them. To persist every epoch to Drive, set `USE_DRIVE=True` and complete Google\'s Drive authorization yourself. The notebook resumes automatically from `latest.pt` when it exists. Keep configuration and dataset paths the same when resuming.\n\n## Local setup\n\nUse Python 3.10+:\n\n```bash\npython -m venv .venv\nsource .venv/bin/activate\npip install -r requirements.txt\npython -m unittest discover -s tests -v\n```\n\nFor Windows, activate with `.venv\\Scripts\\activate`. For CUDA, install the matching Torch/Torchvision build from [PyTorch](https://pytorch.org/get-started/locally/) first. `--device auto` selects CUDA, then Apple MPS, then CPU.\n\n## Data and training\n\n```bash\npython -m chronorestore.download\npython -m chronorestore.train --data data/bsds500/train --val-data data/bsds500/val --output runs/bsds500 --epochs 30\n```\n\nThe [official BSDS500 dataset](https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/resources.html) provides 200 training, 100 validation, and 200 test natural images. It is a small starter dataset, not sufficient evidence of general restoration quality on historical photos. Its original benchmark is image segmentation; our PSNR/SSIM measurements are a separate synthetic restoration experiment.\n\nTo use your own photos, supply separate clean training and validation folders. Supported formats: JPG, PNG, WEBP, BMP, TIFF. Without `--val-data`, a seeded 10% split is reserved automatically. Keep duplicates and near-duplicates out of different splits. Evaluation refuses paths used for training. Synthetic aging uses color fading, warm tint, scratches, dust, noise, JPEG artifacts, and small missing regions.\n\n```bash\npython -m chronorestore.train --data data/bsds500/train --val-data data/bsds500/val --output runs/bsds500 --epochs 60 --resume runs/bsds500/latest.pt\n```\n\n`--epochs` is the total target, not extra epochs. Checkpoints are replaced atomically after each epoch. `latest.pt` includes both networks and optimizers; `best.pt` holds the generator with the lowest validation L1. Existing runs cannot be overwritten accidentally. CSV losses and the image split manifest accompany the checkpoints.\n\nFor an L1-only ablation, use a new output folder and `--gan-weight 0`. Keep the same seed, data, crop size, and epoch budget. Larger datasets and a longer run are needed before claiming useful real-photo restoration.\n\n## Evaluate and restore\n\n```bash\npython -m chronorestore.evaluate --checkpoint runs/bsds500/best.pt --data data/bsds500/test --output evaluation\npython -m chronorestore.restore --checkpoint runs/bsds500/best.pt --input old.jpg --output restored.png\nCHRONORESTORE_CHECKPOINT=runs/bsds500/best.pt python -m chronorestore.demo\n```\n\nEvaluation saves per-image and mean PSNR/SSIM in `metrics.json`, plus comparisons in **damaged / restored / clean** order. SSIM uses scikit-image with RGB channel handling and a data range of 1. Results depend on the fixed degradation seed, crop size, and dataset. Use validation for tuning and reserve test data for final reporting.\n\nInference preserves image dimensions, uses overlapping tiles to limit GPU memory, handles tiny images, and refuses to overwrite an existing output. Tiling can cause subtle local inconsistencies because instance normalization is tile-dependent. Images above 25 megapixels must be resized first.\n\n## Scope and limitations\n\nThis is an educational baseline inspired by [Bringing Old Photos Back to Life](https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life), not a reproduction of its architecture or published results. It does not perform learned super-resolution, explicit face refinement, or real-photo domain adaptation. The original proposed consistency loop could be satisfied without improving restoration, so it was removed. Large missing areas and identity preservation remain difficult; generated detail is approximate. Keep the original photo.\n\nThe code follows the [Ponytail](https://github.com/dietrichgebert/ponytail) preference for a small complete implementation. Tests exercise both networks, deterministic degradation, corrupted-image errors, tiling boundaries, train/resume equivalence, inference, evaluation leakage checks, overwrite protection, and demo construction.\n\nDataset citation: P. Arbelaez, M. Maire, C. Fowlkes, and J. Malik. *Contour Detection and Hierarchical Image Segmentation*. IEEE TPAMI, 2011. Source URL and the downloaded archive\'s SHA-256 are recorded in `data/bsds500/SOURCE.txt`.\n', 'requirements.txt': 'torch>=2.2,<3\ntorchvision>=0.17,<1\nnumpy>=1.26,<3\nPillow>=10,<13\nscikit-image>=0.22,<1\ngradio>=6,<7\nfastapi>=0.115,<1\nuvicorn>=0.30,<1\n', 'requirements-dev.txt': '-r requirements.txt\npytest>=8,<10\n', 'docs/LARGER_TRAINING.md': '# Completed larger training run\n\n30 epochs, 5000 training images, 500 validation images, 256px crops, batch 4 on Tesla T4. Best checkpoint: epoch 28 (validation L1 0.035889).\n\n500 held-out synthetic test images: PSNR 18.54 → 26.37 dB; SSIM 0.7305 → 0.8199. These scores are not directly comparable with the starter benchmark because the test dataset and crop size differ. Real historical photo and face quality still needs review.\n\nLaunch the improved demo from the project directory:\n```sh\nCHRONORESTORE_CHECKPOINT=runs/caltech256_256px/best.pt python -m chronorestore.demo\n```\n\nThe original starter weights are preserved in runs/bsds500. The completed results are saved locally in ChronoRestore_256px_results.zip.\n', 'docs/QUALITY_BENCHMARK.md': '# ChronoRestore Detail benchmark\n\nThe original generator corrects much of the fading and color cast, but can leave scratches, soft fur and missing texture. Detail keeps the same architecture and fine-tunes the original Caltech256 checkpoint for ten epochs on 200 BSDS500 training images. The objective adds horizontal and vertical gradient errors to the pixel L1 loss. The 100 validation images select the epoch with the lowest mean squared error. Epoch 10 reduced validation MSE from 0.0029763 to 0.0023648.\n\n## Controlled comparison\n\nAll rows use the same 200 BSDS500 test images, 256 × 256 center crops, and identical simulated damage per image. The evaluator uses seed 1701 plus the sorted image index. PSNR and SSIM are measured against known clean targets. Edge MAE averages the direction-wise errors between normalized RGB first differences. Refine averages predictions on the input and three flipped versions after reversing each flip.\n\n| Method | Mean PSNR (dB) ↑ | Mean SSIM ↑ | Edge MAE ↓ |\n|---|---:|---:|---:|\n| Damaged input | 18.82 | 0.7275 | 0.02584 |\n| Median + autocontrast | 20.95 | 0.7810 | 0.02631 |\n| Original checkpoint | 25.44 | 0.7934 | 0.02731 |\n| Original + Refine | 26.03 | 0.8232 | 0.02379 |\n| Detail checkpoint | 26.47 | 0.8394 | 0.02253 |\n| **Detail + Refine** | **26.69** | **0.8478** | **0.02182** |\n\nDetail + Refine improved PSNR on 168 images and SSIM on 194 relative to Original + Refine. Mean edge MAE fell by 8.3%. Detail without flips also exceeded Original + Refine on all three mean metrics. This run changed the fine-tuning data and objective together, so it does not isolate the contribution of the gradient term.\n\n## How to use this in the report\n\n> On 200 BSDS500 images with fixed synthetic damage, ChronoRestore Detail + Refine improved mean PSNR from 26.03 to 26.69 dB and mean SSIM from 0.8232 to 0.8478 over the previous Refine checkpoint. Mean edge error decreased by 8.3%. Fine scratches and soft textures remain visible in some outputs.\n\nUse the paired comparison image in `evaluation_detail/paired_comparison.png`. It shows the first three test examples in filename order. The per-image measurements are in `evaluation_detail/metrics.json`. The earlier archive cube and proof-sheet examples still show the original Caltech256 experiment.\n\nWan et al. reported 23.33 dB and 0.69 SSIM on their synthetically degraded DIV2K benchmark. Xu et al. reported 17.20 dB and 0.758 SSIM for joint restoration and colorization on RealOld. Cite these as related work under their own protocols. A model ranking would require rerunning the methods on the same images and scoring protocol. RealOld access is restricted by the authors. This project did not evaluate real scans or audit cross-dataset near duplicates between Caltech256 and BSDS500.\n\n- [Wan et al., CVPR 2020, Table 1](https://openaccess.thecvf.com/content_CVPR_2020/papers/Wan_Bringing_Old_Photos_Back_to_Life_CVPR_2020_paper.pdf)\n- [Xu et al., WACV 2023, Table 2](https://openaccess.thecvf.com/content/WACV2023/papers/Xu_Pik-Fix_Restoring_and_Colorizing_Old_Photos_WACV_2023_paper.pdf)\n- [RealOld access in the official repository](https://github.com/DerrickXuNu/Pik-Fix#real-old-photo-data)\n\n## Reproduce\n\nWith the original Caltech256 checkpoint and local BSDS500 split installed, run `python -m scripts.finetune_detail` in a checkout without an existing `runs/detail_finetune` directory. Then run `python -m scripts.benchmark_detail`. The studio loads the Detail checkpoint by default.\n', 'docs/VERIFICATION.md': '# Verification — 16 October 2026\n\n## Environments exercised\n\n- Local macOS ARM64, Python 3.12, Torch 2.14.1, Torchvision 0.29.1, Gradio 6.30.0. Dependency consistency check passed.\n- Google Colab Python 3, Tesla T4, Torch 2.11.0+cu130, Torchvision 0.26.0+cu130.\n- Live notebook: https://colab.research.google.com/drive/1ssIPq8oVkAbmSdqE-bZ7N9BhILGmpL7O\n\n## Checks completed\n\nAll six regression cases passed locally and in Colab. They cover deterministic degradation, unreadable image errors, generator/discriminator patch sizes, tiny and tiled image boundary handling, invalid settings, and an integrated training/resume/restoration/evaluation/demo flow. The integration case compares resumed generator weights with an uninterrupted run exactly, exercises adversarial backward passes, rejects output overwrites and training-set evaluation, and constructs the demo.\n\nA full 30-epoch CUDA run completed on the official BSDS500 image splits with 128-pixel crops, batch size 8, two warmup epochs, and seed 17. Checkpoint loading through the Colab resume command succeeded after completion. No extra epochs were needed or run for that check.\n\nThe trained checkpoint was downloaded and loaded by the local Gradio server. A client uploaded a 481 × 321 synthetic damaged image, invoked restoration through the running server, and received a valid PNG with the same dimensions. Example files are in `examples/`. The visible browser flow was also exercised: upload the sample through the file chooser, click Restore photo, and confirm the output image and download link appeared. A full-page screenshot is saved as `docs/demo-screenshot.jpg`. Both notebook source cells and ZIP archive integrity were checked.\n\n## Recorded benchmark\n\nModel selection used validation L1 only. The selected checkpoint is epoch 28 (validation L1 0.065344). Evaluation used the 200 separate BSDS500 test images, 128-pixel center crops, and damage seed 1701.\n\n| Metric | Damaged input | Restored output |\n| --- | ---: | ---: |\n| Mean PSNR (dB) | 18.6566 | 21.4322 |\n| Mean SSIM | 0.688835 | 0.723443 |\n\n`evaluation/metrics.json` contains every image score; `runs/bsds500/losses.csv` contains all 30 epochs. The benchmark was rerun unchanged to verify visible notebook output, with identical results. No test scores were used to choose a new training configuration.\n\nThese checks establish the exercised pipeline works. They do not guarantee all possible inputs, future dependency releases, or future Colab sessions will be error-free. Real historical scans were not used for training or quantitative evaluation. Some scratches and missing detail remain; this checkpoint is a starter model, not a finished archival restoration system.\n'}
for name, source in SOURCE_FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)
os.chdir(PROJECT)
print("Project ready:", PROJECT)


In [ ]:
#@title 2. Prepare dependencies and confirm the GPU
def run(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-image>=0.22,<1', 'gradio>=6,<7'])
import torch, torchvision
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime → Change runtime type → T4 GPU, then reconnect.')
print('Torch:', torch.__version__, '| Torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
#@title 3. Run regression checks before training
run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'])


In [ ]:
#@title 4. Download the starter dataset (200 train / 100 validation / 200 test)
run([sys.executable, '-m', 'chronorestore.download'])
DATA = PROJECT / 'data' / 'bsds500'


In [ ]:
#@title 5. Train or resume (T4 defaults)
EPOCHS = 30 #@param {type:"integer"}
IMAGE_SIZE = 128 #@param {type:"integer"}
BATCH_SIZE = 8 #@param {type:"integer"}
USE_DRIVE = False #@param {type:"boolean"}
# Drive is optional. Without it, download the final ZIP before the runtime expires.
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNS = Path('/content/drive/MyDrive/ChronoRestore/runs/bsds500')
else:
    RUNS = PROJECT / 'runs' / 'bsds500'
command = [sys.executable, '-u', '-m', 'chronorestore.train', '--data', str(DATA / 'train'), '--val-data', str(DATA / 'val'), '--output', str(RUNS), '--epochs', str(EPOCHS), '--size', str(IMAGE_SIZE), '--batch-size', str(BATCH_SIZE), '--device', 'cuda']
if (RUNS / 'latest.pt').is_file():
    command += ['--resume', str(RUNS / 'latest.pt')]
run(command)


In [ ]:
#@title 6. Evaluate the selected model on the separate test split
EVALUATION = PROJECT / 'evaluation'
run([sys.executable, '-m', 'chronorestore.evaluate', '--checkpoint', str(RUNS / 'best.pt'), '--data', str(DATA / 'test'), '--output', str(EVALUATION), '--size', str(IMAGE_SIZE), '--count', '200', '--device', 'cuda'])
from IPython.display import display
from PIL import Image
print('Comparisons: damaged / restored / clean')
for path in sorted(EVALUATION.glob('comparison_*.png'))[:3]:
    display(Image.open(path))


In [ ]:
#@title 7. Download trained weights, metrics, and example restorations
import zipfile
from google.colab import files
archive = PROJECT / 'ChronoRestore_results.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RUNS.iterdir()):
        if path.is_file() and path.suffix in ('.pt', '.csv', '.json'):
            bundle.write(path, 'runs/bsds500/' + path.name)
    for path in sorted(EVALUATION.iterdir()):
        if path.is_file():
            bundle.write(path, 'evaluation/' + path.name)
print('Download this before your runtime expires:', archive)
files.download(str(archive))


## Use your trained model
Extract the results ZIP into the local project folder, then run:
```
python -m chronorestore.restore --checkpoint runs/bsds500/best.pt --input old.jpg --output restored.png
```
For the browser demo, set `CHRONORESTORE_CHECKPOINT=runs/bsds500/best.pt` and run `python -m chronorestore.demo` locally. Training data: [official Berkeley BSDS500](https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/resources.html). Keep original photographs; generated details can be approximate.
